In [ ]:
import sys
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator, LogLocator, NullFormatter
from glob import glob
from sklearn.utils import resample

In [ ]:
import pymbar
sys.path.append("../")
from pymbar.mbar_pmf import mbar_pmf

In [ ]:
ene_b3lyp_D3 = np.load('qmmm_6-31g*_D3BJM_energy_new.npy')*627.5
ene_b3lyp_D4 = np.load('qmmm_6-31g*_D4_energy_from_D3BJ_frames.npy')*627.5 

In [ ]:
n_windows = 40
val_kn = []

selected_indices = np.arange(0, 4000, 10) 

for i in range(n_windows):
    fnames = sorted(
        glob('../%02d/step6.00_equilibration.cv' % i) +
        glob('../%02d/step6.01*equilibration.cv' % i)
    )
    
    arrays = [np.loadtxt(f, usecols=1) for f in fnames]
    all_vals = np.concatenate(arrays)
    
    if len(all_vals) < 4000:
        raise ValueError(
            f"Window {i:02d} has only {len(all_vals)} frames — expected at least 4000."
        )
    
    val_kn.append(all_vals[selected_indices])

val_kn = np.array(val_kn)

print("val_kn shape:", val_kn.shape)

val0_k = np.linspace(-1.975, 1.925, n_windows)
K_k = np.ones(n_windows) * 300.0

val_min = -1.975
val_max = 1.925
nbins = n_windows - 1

In [ ]:
for i in range(n_windows):
    print("Window %02d:" % i, pymbar.timeseries.subsampleCorrelatedData(val_kn[i], conservative=True))

In [ ]:
mbar = mbar_pmf(val_kn, val0_k, K_k, 300.0, u_kn=(ene_b3lyp_D3))

In [ ]:
bin_centers, f_i, df_i, reweighting_entropy = mbar.get_pmf(val_min, val_max, nbins, u_kn=ene_b3lyp_D3)
bin_centers, f_i, df_i, reweighting_entropy = mbar.get_pmf(val_min, val_max, nbins, uncertainties='from-specified', pmf_reference=f_i[:20].argmin())

plt.errorbar(bin_centers, f_i - f_i[0], yerr=df_i, linewidth=1, label="B3LYP-D3(BJ)/6-31G*")
plt.legend()

In [ ]:
bin_centers, f_i_rw, df_i_rw, reweighting_entropy = mbar.get_pmf(val_min, val_max, nbins,u_kn=ene_b3lyp_D4, uncertainties='from-specified', pmf_reference=f_i[:20].argmin())
np.savetxt("freefile_mbar_b3lyp_d", np.column_stack((bin_centers, reweighting_entropy)), fmt="%.2f %f")

plt.xlabel("Reaction Coordinate (Å)", fontsize=20)
plt.ylabel("Potential of Mean Force (kcal/mol)", fontsize=20)
plt.errorbar(bin_centers, f_i - f_i[0], yerr=df_i, linewidth=2.0, label="B3LYP-D3(BJ)/6-31G* (Ref)")
plt.errorbar(bin_centers, f_i_rw - f_i_rw[0], yerr=df_i_rw, linewidth=2.0, label="B3LYP-D4/6-31G*")
plt.tight_layout()
plt.legend(fontsize=9, loc=3)

In [ ]:
bin_centers, f_i, df_i, reweighting_entropy = mbar.get_pmf(val_min, val_max, nbins, u_kn=ene_b3lyp_D4)
bin_centers, f_i, df_i, reweighting_entropy = mbar.get_pmf(val_min, val_max, nbins, uncertainties='from-specified', pmf_reference=f_i[:20].argmin())

plt.errorbar(bin_centers, f_i - f_i[0], yerr=df_i, linewidth=1, label="B3LYP-D4/6-31G*")
plt.legend()
np.savetxt(f"reweighting_entropy_D4_new", reweighting_entropy)

In [ ]:
# =========================================================
# Reweighted PMF: B3LYP-D4
# =========================================================

bin_centers, f_i_rw, df_i_rw, reweighting_entropy = mbar.get_pmf(
    val_min,
    val_max,
    nbins,
    u_kn=ene_b3lyp_D4,
    uncertainties='from-specified',
    pmf_reference=f_i[:20].argmin()
)

np.savetxt(
    "freefile_mbar_b3lyp_d",
    np.column_stack((bin_centers, reweighting_entropy)),
    fmt="%.2f %f"
)

# =========================================================
# Calculate barriers
# =========================================================

# Reactant-side region
reactant_bins = np.arange(20)

# D3(BJ) reference PMF
r_d3 = reactant_bins[np.argmin(f_i[reactant_bins])]
ts_d3 = np.argmax(f_i[r_d3:]) + r_d3

barrier_d3 = f_i[ts_d3] - f_i[r_d3]
barrier_err_d3 = np.sqrt(df_i[ts_d3]**2 + df_i[r_d3]**2)

# D4 reweighted PMF
r_d4 = reactant_bins[np.argmin(f_i_rw[reactant_bins])]
ts_d4 = np.argmax(f_i_rw[r_d4:]) + r_d4

barrier_d4 = f_i_rw[ts_d4] - f_i_rw[r_d4]
barrier_err_d4 = np.sqrt(df_i_rw[ts_d4]**2 + df_i_rw[r_d4]**2)

print(f"B3LYP-D3(BJ) barrier = {barrier_d3:.2f} ± {barrier_err_d3:.2f} kcal/mol")
print(f"B3LYP-D4 barrier     = {barrier_d4:.2f} ± {barrier_err_d4:.2f} kcal/mol")

# =========================================================
# Plot PMFs with barriers in legend
# =========================================================

plt.figure(figsize=(7, 5))

plt.xlabel("Reaction Coordinate (Å)", fontsize=14)
plt.ylabel("Potential of Mean Force (kcal/mol)", fontsize=14)

plt.errorbar(
    bin_centers,
    f_i - f_i[r_d3],
    yerr=df_i,
    linewidth=2.0,
    label=f"B3LYP-D3(BJ)/6-31G*: {barrier_d3:.2f} ± {barrier_err_d3:.2f} kcal/mol"
)

plt.errorbar(
    bin_centers,
    f_i_rw - f_i_rw[r_d4],
    yerr=df_i_rw,
    linewidth=2.0,
    label=f"B3LYP-D4/6-31G*: {barrier_d4:.2f} ± {barrier_err_d4:.2f} kcal/mol"
)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_ticks_position("bottom")
ax.yaxis.set_ticks_position("left")

plt.legend(fontsize=9, loc=3, frameon=False)
plt.tight_layout()

plt.savefig("pmf_D3BJ_vs_D4_new.png", dpi=300, bbox_inches="tight")
plt.savefig("pmf_D3BJ_vs_D4_new.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# =========================================================
# ONE-COLUMN PMF PLOT: D3(BJ) vs D4
# =========================================================

# Exact final PDF dimensions
FIGURE_WIDTH = 3.50
FIGURE_HEIGHT = 3.00

# Exact plotting-area dimensions
PLOT_WIDTH = 2.78
PLOT_HEIGHT = 2.22

PLOT_LEFT = 0.56
PLOT_BOTTOM = 0.50

# Distinct, colorblind-friendly colors
cmap = plt.get_cmap("tab20")

color_d3 = cmap(6) 
color_d4 = cmap(0)  

# =========================================================
# CREATE EXACT-SIZE FIGURE
# =========================================================

fig = plt.figure(
    figsize=(FIGURE_WIDTH, FIGURE_HEIGHT),
    facecolor="white"
)

ax = fig.add_axes([
    PLOT_LEFT / FIGURE_WIDTH,
    PLOT_BOTTOM / FIGURE_HEIGHT,
    PLOT_WIDTH / FIGURE_WIDTH,
    PLOT_HEIGHT / FIGURE_HEIGHT
])

# =========================================================
# PLOT PMFs
# =========================================================
# D3(BJ): solid line with open circles
ax.errorbar(
    bin_centers,
    f_i - f_i[r_d3],
    yerr=df_i,
    color=color_d3,
    linewidth=1.5,
    linestyle="-",
    elinewidth=0.55,
    errorevery=(0, 5),
    capsize=0,
    zorder=2
)

# D4: dashed line with open squares
ax.errorbar(
    bin_centers,
    f_i_rw - f_i_rw[r_d4],
    yerr=df_i_rw,
    color=color_d4,
    linewidth=1.5,
    linestyle=(0, (4, 2)),
    elinewidth=0.55,
    errorevery=(2, 5),
    capsize=0,
    zorder=3
)
# =========================================================
# BARRIER VALUES — LOWER-LEFT CORNER
# =========================================================

ax.text(
    0.04,
    0.04,
    f"D3(BJ): {barrier_d3:.2f} ± {barrier_err_d3:.2f} kcal/mol",
    transform=ax.transAxes,
    fontsize=6,
    fontweight="bold",
    family="monospace",
    color=color_d3,
    horizontalalignment="left",
    verticalalignment="bottom"
)

ax.text(
    0.04,
    0.115,
    f"D4    : {barrier_d4:.2f} ± {barrier_err_d4:.2f} kcal/mol",
    transform=ax.transAxes,
    fontsize=6,
    fontweight="bold",
    family="monospace",
    color=color_d4,
    horizontalalignment="left",
    verticalalignment="bottom"
)

# =========================================================
# AXES FORMATTING
# =========================================================

ax.set_xlabel(
    "Reaction Coordinate (Å)",
    fontsize=8,
    labelpad=3
)

ax.set_ylabel(
    "Potential of Mean Force (kcal/mol)",
    fontsize=8,
    labelpad=3
)

ax.tick_params(
    axis="both",
    which="major",
    labelsize=5.5,
    width=0.7,
    length=2.5,
    direction="out"
)

ax.set_xlim(-2, 2)
ax.set_ylim(-20, 20)
ax.grid(False)
ax.set_yticks(np.arange(-20, 16, 5))


ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

ax.spines["bottom"].set_linewidth(0.7)
ax.spines["left"].set_linewidth(0.7)

ax.xaxis.set_ticks_position("bottom")
ax.yaxis.set_ticks_position("left")

# =========================================================
# SAVE EXACT-SIZE VECTOR PDF
# =========================================================

fig.savefig(
    "pmf_D3BJ_vs_D4_one_column.pdf",
    format="pdf",
    facecolor="white"
)

print(
    f"Final PDF size: "
    f"{fig.get_figwidth():.2f} × "
    f"{fig.get_figheight():.2f} inches"
)

plt.show()

In [ ]:
# ============================================================
# CALCULATE D4 REWEIGHTING ENTROPY
# ============================================================

bin_centers, f_i_D4, df_i_D4, reweighting_entropy_D4 = mbar.get_pmf(
    val_min,
    val_max,
    nbins,
    u_kn=ene_b3lyp_D4,
    uncertainties="from-specified",
    pmf_reference=f_i[:20].argmin()
)

print("bin_centers shape:", bin_centers.shape)
print("reweighting_entropy_D4 shape:", reweighting_entropy_D4.shape)


# ============================================================
# SAVE REWEIGHTING ENTROPY
# ============================================================

np.savetxt(
    "reweighting_entropy_D4_new.dat",
    np.column_stack((bin_centers, reweighting_entropy_D4)),
    header="Reaction_coordinate_Angstrom  Reweighting_entropy",
    fmt="%.6f"
)


# ============================================================
# PLOT REWEIGHTING ENTROPY AS A BAR CHART
# ============================================================

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

fig, ax = plt.subplots(figsize=(3.5, 3.0))

# Width based on the spacing between neighboring reaction-coordinate bins
bar_width = 0.8 * np.mean(np.diff(bin_centers))

ax.bar(
    bin_centers,
    reweighting_entropy_D4,
    width=bar_width,
    edgecolor="black",
    linewidth=0.5
)

ax.set_xlabel(r"Reaction Coordinate ($\mathrm{\AA}$)")
ax.set_ylabel("Reweighting Entropy")

ax.set_xlim(-2.0, 2.0)
ax.set_xticks(np.arange(-2.0, 2.1, 0.5))

# Make the y-axis begin at zero
ax.set_ylim(bottom=0)

ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    length=4,
    width=0.8
)

ax.xaxis.set_minor_locator(AutoMinorLocator(2))
ax.yaxis.set_minor_locator(AutoMinorLocator(2))

ax.tick_params(
    axis="both",
    which="minor",
    direction="out",
    length=2,
    width=0.6
)

fig.tight_layout()

fig.savefig(
    "reweighting_entropy_D4_new.pdf",
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================================
# REWEIGHTING ENTROPY BAR PLOT

# ============================================================

# ============================================================
# GENERAL PLOT SETTINGS
# ============================================================

# Keep text editable when the figure is opened in Illustrator
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["axes.grid"] = False
plt.rcParams["axes.xmargin"] = 0.05
plt.rcParams["axes.ymargin"] = 0.10


# ============================================================
# IF NEEDED: CALCULATE REWEIGHTING ENTROPY FOR D4
# ============================================================

bin_centers, f_i_D4, df_i_D4, reweighting_entropy_D4 = mbar.get_pmf(
    val_min,
    val_max,
    nbins,
    u_kn=ene_b3lyp_D4,
    uncertainties="from-specified",
    pmf_reference=f_i[:20].argmin()
)

# Save data if you want
np.savetxt(
    "reweighting_entropy_D4_new.dat",
    np.column_stack((bin_centers, reweighting_entropy_D4)),
    fmt="%.6f"
)


# ============================================================
# FULL-WIDTH MANUSCRIPT FIGURE SIZE
# ============================================================

FIGURE_WIDTH = 7.20
FIGURE_HEIGHT = 2.80


# ============================================================
# PLOTTING-AREA DIMENSIONS
# ============================================================

PLOT_LEFT = 0.65
PLOT_BOTTOM = 0.55
PLOT_WIDTH = 6.25
PLOT_HEIGHT = 1.85


# ============================================================
# CREATE EXACT-SIZE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(FIGURE_WIDTH, FIGURE_HEIGHT),
    facecolor="white"
)

ax = fig.add_axes([
    PLOT_LEFT / FIGURE_WIDTH,
    PLOT_BOTTOM / FIGURE_HEIGHT,
    PLOT_WIDTH / FIGURE_WIDTH,
    PLOT_HEIGHT / FIGURE_HEIGHT
])


# ============================================================
# BAR POSITIONS, WIDTH, AND SPACING
# ============================================================

# Larger spacing between bars
SPACING_FACTOR = 1.45

# Larger bar width
BAR_WIDTH = 1.10

xpos = np.arange(len(bin_centers)) * SPACING_FACTOR


# ============================================================
# PLOT BARS
# ============================================================

ax.bar(
    xpos,
    reweighting_entropy_D4,
    width=BAR_WIDTH,
    edgecolor="black",
    linewidth=0.6
)


# ============================================================
# AXIS LABELS
# ============================================================

ax.set_xlabel(r"Reaction Coordinate ($\mathrm{\AA}$)")
ax.set_ylabel("Reweighting Entropy")


# ============================================================
# X TICKS
# ============================================================

# Major ticks: show labels every 4th bar
tick_step = 4
tick_indices = np.arange(0, len(bin_centers), tick_step)

ax.set_xticks(xpos[tick_indices])
ax.set_xticklabels(
    [f"{bin_centers[i]:.1f}" for i in tick_indices],
    rotation=45,
    ha="right"
)

# Minor ticks: every bar position, for vertical grid lines
ax.set_xticks(xpos, minor=True)


# ============================================================
# AXIS LIMITS
# ============================================================

ax.set_xlim(xpos[0] - BAR_WIDTH, xpos[-1] + BAR_WIDTH)
ax.set_ylim(bottom=0)


# ============================================================
# GRID BEHIND BARS
# ============================================================

ax.set_axisbelow(True)
ax.set_facecolor("0.95")

# Horizontal grid lines
ax.grid(
    True,
    which="major",
    axis="y",
    linestyle="--",
    linewidth=0.8,
    color="0.82"
)



# ============================================================
# TICKS
# ============================================================

ax.tick_params(
    axis="both",
    which="major",
    direction="out",
    length=4,
    width=0.8
)

ax.tick_params(
    axis="both",
    which="minor",
    direction="out",
    length=2,
    width=0.6
)

ax.yaxis.set_minor_locator(AutoMinorLocator(2))


# ============================================================
# SAVE FIGURE
# ============================================================

fig.savefig(
    "reweighting_entropy_D4_fullwidth_grid.pdf",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# IF PMF IS NOT SMOOTH, THIS CODE USES GPR TO SMOOTHEN
likelihood_all = []
for i in range(300):
    alpha = (i + 1) * 0.01
    y_pred, sigma, likelihood = mbar.get_reweighted_tp(bin_centers, f_i_rw, reweighting_entropy, alpha)
    likelihood_all.append(likelihood)
print(np.array(likelihood_all).max(), np.array(likelihood_all).argmax())

In [ ]:
y_pred, sigma, likelihood = mbar.get_reweighted_tp(bin_centers, f_i_rw, reweighting_entropy, 0.57)
sigma2 = sigma + sigma[y_pred[:10].argmin()]
sigma2[y_pred[:10].argmin()] = 0.0

plt.xlabel("Reaction Coordinate ($\AA$)", fontsize=14)
plt.ylabel("Potential of Mean Force (kcal/mol)", fontsize=14)

# Reference
plt.errorbar(
    bin_centers,
    f_i - f_i[:10].min(),
    yerr=df_i,
    linewidth=2.0,
    label="B3LYP/6-31G* (Ref) (%.1f ± %.1f)" % (
        f_i.max() - f_i[:10].min(),
        df_i[f_i.argmax()]
    )
)

# Reweighted
plt.errorbar(
    bin_centers,
    y_pred - y_pred[:10].min(),
    yerr=sigma2,
    linewidth=2.0,
    label=r"B3LYP-D3BJ/6-31G* (%.1f ± %.1f)" % (
        y_pred.max() - y_pred[:10].min(),
        sigma2[y_pred.argmax()]
    )
)

plt.legend(fontsize=9, loc=3)
plt.tight_layout()

print("Barrier Height (kcal/mol): %.2f" % (y_pred.max() - y_pred[:10].min()))

np.savetxt("freefile_mbar_b3lyp_d_gpr", np.column_stack((bin_centers, y_pred, sigma2)), fmt="%.2f %f %f")
plt.savefig("pmf_reweighted_b3lyp_d3bjm.png", dpi=300)